# PhoBERT Multi-task Training

Notebook nay train mot model PhoBERT dung chung encoder cho 2 task:

- sentiment classification
- topic classification

Notebook chay duoc tren Google Colab va gom luon phan evaluate tren test set.

## 1. Setup Colab

In [1]:
!pip -q install transformers accelerate underthesea scikit-learn pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 49.3 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import json
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

from google.colab import drive
drive.mount('/content/drive')

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

Mounted at /content/drive


device(type='cuda')

## 2. Config paths

Neu chay tren Colab, dat `PROJECT_DIR` toi thu muc repo tren Drive. Sua bien ben duoi neu duong dan cua ban khac.

In [3]:
PROJECT_DIR = Path('/content/drive/MyDrive/Student Voice Intelligence')

DATA_PATH = PROJECT_DIR / 'data/processed/student_voice_enriched.csv'
MODEL_NAME = 'vinai/phobert-base-v2'
OUTPUT_DIR = PROJECT_DIR / 'outputs/models/phobert_multitask'
REPORT_DIR = PROJECT_DIR / 'outputs/reports/transformer/phobert_multitask'
CACHE_PATH = PROJECT_DIR / 'data/processed/student_voice_enriched_multitask_segmented.csv'

MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
SENTIMENT_LOSS_WEIGHT = 1.0
TOPIC_LOSS_WEIGHT = 1.0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Project:', PROJECT_DIR)
print('Data:', DATA_PATH)
print('Device:', DEVICE)

Project: /content/drive/MyDrive/Student Voice Intelligence
Data: /content/drive/MyDrive/Student Voice Intelligence/data/processed/student_voice_enriched.csv
Device: cuda


## 3. Load and prepare data

In [4]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
print(df.columns.tolist())
df.head(3)

(49141, 19)
['text', 'source_dataset', 'split_original', 'sentiment_raw', 'sentiment_std_code', 'sentiment_std_3class', 'sentiment_std_4class', 'topic_raw', 'topic_std', 'is_toxic', 'urgency_level', 'is_toxic_seed', 'toxic_rule_hits', 'is_toxic_rule', 'toxic_label_source', 'urgency_rule_hits', 'urgency_label_source', 'topic_group', 'label_needs_review']


,text,source_dataset,split_original,sentiment_raw,sentiment_std_code,sentiment_std_3class,sentiment_std_4class,topic_raw,topic_std,is_toxic,urgency_level,is_toxic_seed,toxic_rule_hits,is_toxic_rule,toxic_label_source,urgency_rule_hits,urgency_label_source,topic_group,label_needs_review
0,em xin chào các anh chị em em là sinh viên vừa...,NEU_ESC,train,0,0,neutral,neutral,2,academic,0,low,0,NaN,0,default_non_toxic,NaN,default_low,teaching_learning,0
1,xây dựng mô hình sư phạm chuẩn mực sáng tạo ti...,NEU_ESC,train,0,0,neutral,neutral,2,academic,0,low,0,NaN,0,default_non_toxic,NaN,default_low,teaching_learning,0
2,sao lại ghét cái kiểu làm sai xong khóc lóc ăn...,NEU_ESC,train,2,2,negative,negative,3,others,0,low,0,NaN,0,default_non_toxic,NaN,default_low,others,0


In [5]:
TEXT_COL = 'text'
SENTIMENT_COL = 'sentiment_std_3class'
TOPIC_COL = 'topic_group' if 'topic_group' in df.columns else 'topic_std'
SPLIT_COL = 'split_original'

required = [TEXT_COL, SENTIMENT_COL, TOPIC_COL, SPLIT_COL]
missing = [col for col in required if col not in df.columns]

work = df[required].dropna().copy()
work[TEXT_COL] = work[TEXT_COL].astype(str).str.strip()
work[SENTIMENT_COL] = work[SENTIMENT_COL].astype(str).str.strip()
work[TOPIC_COL] = work[TOPIC_COL].astype(str).str.strip()
work[SPLIT_COL] = work[SPLIT_COL].astype(str).str.lower().str.strip()
work = work[work[TEXT_COL] != ''].copy()

split_map = {'val': 'validation', 'valid': 'validation', 'dev': 'validation'}
work[SPLIT_COL] = work[SPLIT_COL].replace(split_map)
print(work[SPLIT_COL].value_counts())
print(work[SENTIMENT_COL].value_counts())
print(work[TOPIC_COL].value_counts())

split_original
train         34474
test           9779
validation     4888
Name: count, dtype: int64
sentiment_std_3class
neutral     23471
negative    13484
positive    12186
Name: count, dtype: int64
topic_group
teaching_learning    25159
others               15218
student_services      3028
personal_social       2247
events_news           1564
career_jobs            808
facilities             712
spam                   405
Name: count, dtype: int64


## 4. Word segmentation for PhoBERT

PhoBERT cho ket qua tot hon khi text da duoc word-segment. Cell nay cache ket qua de lan sau chay nhanh hon.

In [6]:
if CACHE_PATH.exists():
    work = pd.read_csv(CACHE_PATH)
    print('Loaded cache:', CACHE_PATH, work.shape)
else:
    from underthesea import word_tokenize
    tqdm.pandas(desc='Word segmentation')
    work['segmented_text'] = work[TEXT_COL].progress_apply(lambda text: word_tokenize(text, format='text'))
    work.to_csv(CACHE_PATH, index=False)
    print('Saved cache:', CACHE_PATH, work.shape)

work[['text', 'segmented_text', SENTIMENT_COL, TOPIC_COL, SPLIT_COL]].head(3)

Word segmentation:   0%|          | 0/49141 [00:00<?, ?it/s]

Saved cache: /content/drive/MyDrive/Student Voice Intelligence/data/processed/student_voice_enriched_multitask_segmented.csv (49141, 5)


,text,segmented_text,sentiment_std_3class,topic_group,split_original
0,em xin chào các anh chị em em là sinh viên vừa...,em xin chào các anh_chị_em em là sinh_viên vừa...,neutral,teaching_learning,train
1,xây dựng mô hình sư phạm chuẩn mực sáng tạo ti...,xây_dựng mô_hình sư_phạm chuẩn_mực sáng_tạo ti...,neutral,teaching_learning,train
2,sao lại ghét cái kiểu làm sai xong khóc lóc ăn...,sao lại ghét cái kiểu làm sai xong khóc_lóc ăn...,negative,others,train


## 5. Label mappings and datasets

In [7]:
sentiment_labels = sorted(work[SENTIMENT_COL].unique().tolist())
topic_labels = sorted(work[TOPIC_COL].unique().tolist())

sentiment2id = {label: idx for idx, label in enumerate(sentiment_labels)}
id2sentiment = {idx: label for label, idx in sentiment2id.items()}
topic2id = {label: idx for idx, label in enumerate(topic_labels)}
id2topic = {idx: label for label, idx in topic2id.items()}

work['sentiment_label'] = work[SENTIMENT_COL].map(sentiment2id)
work['topic_label'] = work[TOPIC_COL].map(topic2id)

label_config = {
    'model_name': MODEL_NAME,
    'text_col': TEXT_COL,
    'sentiment_col': SENTIMENT_COL,
    'topic_col': TOPIC_COL,
    'sentiment2id': sentiment2id,
    'id2sentiment': {str(k): v for k, v in id2sentiment.items()},
    'topic2id': topic2id,
    'id2topic': {str(k): v for k, v in id2topic.items()},
    'max_length': MAX_LENGTH,
}
(OUTPUT_DIR / 'multitask_config.json').write_text(json.dumps(label_config, ensure_ascii=False, indent=2), encoding='utf-8')
print('sentiment labels:', sentiment2id)
print('topic labels:', topic2id)

sentiment labels: {'negative': 0, 'neutral': 1, 'positive': 2}
topic labels: {'career_jobs': 0, 'events_news': 1, 'facilities': 2, 'others': 3, 'personal_social': 4, 'spam': 5, 'student_services': 6, 'teaching_learning': 7}


In [8]:
train_df = work[work[SPLIT_COL] == 'train'].reset_index(drop=True)
val_df = work[work[SPLIT_COL].isin(['validation'])].reset_index(drop=True)
test_df = work[work[SPLIT_COL] == 'test'].reset_index(drop=True)

print('train:', train_df.shape)
print('val:', val_df.shape)
print('test:', test_df.shape)

train: (34474, 7)
val: (4888, 7)
test: (9779, 7)


In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

class MultiTaskFeedbackDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tokenizer, max_length: int):
        self.texts = frame['segmented_text'].astype(str).tolist()
        self.sentiment_labels = frame['sentiment_label'].astype(int).tolist()
        self.topic_labels = frame['topic_label'].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoded = self.tokenizer(
            self.texts[index],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'sentiment_labels': torch.tensor(self.sentiment_labels[index], dtype=torch.long),
            'topic_labels': torch.tensor(self.topic_labels[index], dtype=torch.long),
        }

train_loader = DataLoader(MultiTaskFeedbackDataset(train_df, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(MultiTaskFeedbackDataset(val_df, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(MultiTaskFeedbackDataset(test_df, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

## 6. Multi-task PhoBERT model

In [10]:
class PhoBERTMultiTask(nn.Module):
    def __init__(self, model_name: str, num_sentiment_labels: int, num_topic_labels: int, dropout: float = 0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.sentiment_head = nn.Linear(hidden_size, num_sentiment_labels)
        self.topic_head = nn.Linear(hidden_size, num_topic_labels)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask, sentiment_labels=None, topic_labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0]
        pooled = self.dropout(cls_embedding)
        sentiment_logits = self.sentiment_head(pooled)
        topic_logits = self.topic_head(pooled)

        loss = None
        sentiment_loss = None
        topic_loss = None
        if sentiment_labels is not None and topic_labels is not None:
            sentiment_loss = self.loss_fn(sentiment_logits, sentiment_labels)
            topic_loss = self.loss_fn(topic_logits, topic_labels)
            loss = SENTIMENT_LOSS_WEIGHT * sentiment_loss + TOPIC_LOSS_WEIGHT * topic_loss

        return {
            'loss': loss,
            'sentiment_loss': sentiment_loss,
            'topic_loss': topic_loss,
            'sentiment_logits': sentiment_logits,
            'topic_logits': topic_logits,
        }

model = PhoBERTMultiTask(MODEL_NAME, len(sentiment2id), len(topic2id)).to(DEVICE)
sum(p.numel() for p in model.parameters())

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

135006731

## 7. Train and validate

In [11]:
def evaluate(model, loader):
    model.eval()
    losses = []
    sentiment_true, sentiment_pred = [], []
    topic_true, topic_pred = [], []
    with torch.no_grad():
        for batch in tqdm(loader, leave=False):
            batch = {key: value.to(DEVICE) for key, value in batch.items()}
            outputs = model(**batch)
            if outputs['loss'] is not None:
                losses.append(outputs['loss'].item())
            sentiment_true.extend(batch['sentiment_labels'].cpu().numpy().tolist())
            topic_true.extend(batch['topic_labels'].cpu().numpy().tolist())
            sentiment_pred.extend(outputs['sentiment_logits'].argmax(dim=-1).cpu().numpy().tolist())
            topic_pred.extend(outputs['topic_logits'].argmax(dim=-1).cpu().numpy().tolist())
    return {
        'loss': float(np.mean(losses)) if losses else 0.0,
        'sentiment_accuracy': accuracy_score(sentiment_true, sentiment_pred),
        'sentiment_macro_f1': f1_score(sentiment_true, sentiment_pred, average='macro'),
        'sentiment_weighted_f1': f1_score(sentiment_true, sentiment_pred, average='weighted'),
        'topic_accuracy': accuracy_score(topic_true, topic_pred),
        'topic_macro_f1': f1_score(topic_true, topic_pred, average='macro'),
        'topic_weighted_f1': f1_score(topic_true, topic_pred, average='weighted'),
        'sentiment_true': sentiment_true,
        'sentiment_pred': sentiment_pred,
        'topic_true': topic_true,
        'topic_pred': topic_pred,
    }

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')

history = []
best_score = -1.0
best_path = OUTPUT_DIR / 'best_model.pt'

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []
    started = time.time()
    for batch in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}'):
        batch = {key: value.to(DEVICE) for key, value in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
            outputs = model(**batch)
            loss = outputs['loss']
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        train_losses.append(loss.item())

    val_metrics = evaluate(model, val_loader)
    combined_score = (val_metrics['sentiment_macro_f1'] + val_metrics['topic_macro_f1']) / 2
    row = {
        'epoch': epoch,
        'train_loss': float(np.mean(train_losses)),
        'val_loss': val_metrics['loss'],
        'val_sentiment_macro_f1': val_metrics['sentiment_macro_f1'],
        'val_topic_macro_f1': val_metrics['topic_macro_f1'],
        'val_combined_macro_f1': combined_score,
        'seconds': round(time.time() - started, 1),
    }
    history.append(row)
    print(row)
    if combined_score > best_score:
        best_score = combined_score
        torch.save(model.state_dict(), best_path)
        print('Saved best model:', best_path)

pd.DataFrame(history).to_csv(REPORT_DIR / 'training_history.csv', index=False)
pd.DataFrame(history)

/tmp/ipykernel_3383/3121546496.py:34: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')


Epoch 1/3:   0%|          | 0/2155 [00:00<?, ?it/s]

/tmp/ipykernel_3383/3121546496.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):


  0%|          | 0/306 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 1.231984914981019, 'val_loss': 0.9000490778704094, 'val_sentiment_macro_f1': 0.85752684261818, 'val_topic_macro_f1': 0.6715014564224893, 'val_combined_macro_f1': 0.7645141495203347, 'seconds': 344.8}
Saved best model: /content/drive/MyDrive/Student Voice Intelligence/outputs/models/phobert_multitask/best_model.pt


...

Epoch 3/3:   0%|          | 0/2155 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d302d0d3b00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
    Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7d302d0d3b00> 
 Traceback (most recent call last):
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    ^^self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
^    ^if w.is_alive():^
 ^^ 
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
     assert self._parent_pid == os.getpid(), 'can only test a child process' 
     ^ ^ ^ ^^^  ^ ^^ ^ ^ ^^^
  File "/u

  0%|          | 0/306 [00:00<?, ?it/s]

{'epoch': 3, 'train_loss': 0.6026097768973862, 'val_loss': 0.8369536794770777, 'val_sentiment_macro_f1': 0.8671308676870645, 'val_topic_macro_f1': 0.7344917529365405, 'val_combined_macro_f1': 0.8008113103118024, 'seconds': 371.4}
Saved best model: /content/drive/MyDrive/Student Voice Intelligence/outputs/models/phobert_multitask/best_model.pt


,epoch,train_loss,val_loss,val_sentiment_macro_f1,val_topic_macro_f1,val_combined_macro_f1,seconds
0,1,1.231985,0.900049,0.857527,0.671501,0.764514,344.8
1,2,0.769382,0.829626,0.866987,0.728676,0.797832,344.8
2,3,0.602610,0.836954,0.867131,0.734492,0.800811,371.4


## 8. Test evaluation

In [12]:
model.load_state_dict(torch.load(OUTPUT_DIR / 'best_model.pt', map_location=DEVICE))
test_metrics = evaluate(model, test_loader)

sentiment_target_names = [id2sentiment[i] for i in range(len(id2sentiment))]
topic_target_names = [id2topic[i] for i in range(len(id2topic))]

sentiment_report = classification_report(
    test_metrics['sentiment_true'],
    test_metrics['sentiment_pred'],
    labels=list(range(len(sentiment_target_names))),
    target_names=sentiment_target_names,
    output_dict=True,
    zero_division=0,
)
topic_report = classification_report(
    test_metrics['topic_true'],
    test_metrics['topic_pred'],
    labels=list(range(len(topic_target_names))),
    target_names=topic_target_names,
    output_dict=True,
    zero_division=0,
)

sentiment_cm = confusion_matrix(test_metrics['sentiment_true'], test_metrics['sentiment_pred'], labels=list(range(len(sentiment_target_names))))
topic_cm = confusion_matrix(test_metrics['topic_true'], test_metrics['topic_pred'], labels=list(range(len(topic_target_names))))

summary = {
    'model': 'PhoBERT-base-v2 multi-task',
    'test_loss': test_metrics['loss'],
    'sentiment_accuracy': test_metrics['sentiment_accuracy'],
    'sentiment_macro_f1': test_metrics['sentiment_macro_f1'],
    'sentiment_weighted_f1': test_metrics['sentiment_weighted_f1'],
    'topic_accuracy': test_metrics['topic_accuracy'],
    'topic_macro_f1': test_metrics['topic_macro_f1'],
    'topic_weighted_f1': test_metrics['topic_weighted_f1'],
    'sentiment_labels': sentiment_target_names,
    'topic_labels': topic_target_names,
}

(REPORT_DIR / 'metrics.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
pd.DataFrame(sentiment_report).transpose().to_csv(REPORT_DIR / 'test_classification_report_sentiment.csv')
pd.DataFrame(topic_report).transpose().to_csv(REPORT_DIR / 'test_classification_report_topic.csv')
pd.DataFrame(sentiment_cm, index=sentiment_target_names, columns=sentiment_target_names).to_csv(REPORT_DIR / 'test_confusion_matrix_sentiment.csv')
pd.DataFrame(topic_cm, index=topic_target_names, columns=topic_target_names).to_csv(REPORT_DIR / 'test_confusion_matrix_topic.csv')

summary

  0%|          | 0/612 [00:00<?, ?it/s]

{'model': 'PhoBERT-base-v2 multi-task',
 'test_loss': 0.8780632355353899,
 'sentiment_accuracy': 0.8591880560384497,
 'sentiment_macro_f1': 0.8571483325931418,
 'sentiment_weighted_f1': 0.859203754081548,
 'topic_accuracy': 0.8463033029962164,
 'topic_macro_f1': 0.707159796419521,
 'topic_weighted_f1': 0.8432084268664453,
 'sentiment_labels': ['negative', 'neutral', 'positive'],
 'topic_labels': ['career_jobs',
  'events_news',
  'facilities',
  'others',
  'personal_social',
  'spam',
  'student_services',
  'teaching_learning']}

## 9. Save model artifacts

In [13]:
torch.save(model.state_dict(), OUTPUT_DIR / 'pytorch_model.bin')
tokenizer.save_pretrained(OUTPUT_DIR)
(OUTPUT_DIR / 'multitask_config.json').write_text(json.dumps(label_config, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved model to:', OUTPUT_DIR)
print('Saved reports to:', REPORT_DIR)

Saved model to: /content/drive/MyDrive/Student Voice Intelligence/outputs/models/phobert_multitask
Saved reports to: /content/drive/MyDrive/Student Voice Intelligence/outputs/reports/transformer/phobert_multitask


## 10. Optional comparison with single-task reports

In [14]:
comparison_rows = [
    {'model': 'PhoBERT-base-v2 multi-task', 'task': 'sentiment', 'accuracy': summary['sentiment_accuracy'], 'macro_f1': summary['sentiment_macro_f1'], 'weighted_f1': summary['sentiment_weighted_f1']},
    {'model': 'PhoBERT-base-v2 multi-task', 'task': 'topic', 'accuracy': summary['topic_accuracy'], 'macro_f1': summary['topic_macro_f1'], 'weighted_f1': summary['topic_weighted_f1']},
]
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(REPORT_DIR / 'multitask_summary.csv', index=False)
comparison

,model,task,accuracy,macro_f1,weighted_f1
0,PhoBERT-base-v2 multi-task,sentiment,0.859188,0.857148,0.859204
1,PhoBERT-base-v2 multi-task,topic,0.846303,0.707160,0.843208
